In [95]:
import torch
from torch.utils.data import DataLoader
import numpy as np
import json
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
from dataset import create_dataloaders

# Keypoint indices
LEFT_SHOULDER = 5
RIGHT_SHOULDER = 6
LEFT_HIP = 11
RIGHT_HIP = 12

# Thresholds (coordinates are normalised)
HIP_DESCENT_THRESHOLD = 0.1 # Rule 1: hip descent across full clip (from average in first 10 frames to average in last 10 frames)
BODY_HORIZONTAL_THRESHOLD = 0.12 # Rule 2: shoulder to hip vertical gap average in final 10 frames
RAPID_DESCENT_THRESHOLD = 0.02 # Rule 3: hip descent in any 10-frame period

def classify_sequence(sequence):
    # Extract hip Y coordinates midpoint in each frame
    hip_y = (sequence[:, LEFT_HIP, 1] + sequence[:, RIGHT_HIP, 1]) / 2
    
    # Extract shoulder Y coordinates midpoint in each frame
    shoulder_y = (sequence[:, LEFT_SHOULDER, 1] + sequence[:, RIGHT_SHOULDER, 1]) / 2
    
    # Rule 1: Overall hip descent
    hip_y_start = np.mean(hip_y[:10]) # Average hip Y in first 10 frames
    hip_y_end = np.mean(hip_y[-10:]) # Average hip Y in last 10 frames
    rule1 = (hip_y_end - hip_y_start) > HIP_DESCENT_THRESHOLD # Rule 1 = True if difference from start to end is higher than the set threshold
    
    # Rule 2: Body orientation (check if body is horizontal)
    shoulder_y_end = np.mean(shoulder_y[-10:])  # Average shoulder Y in last 10 frames
    vertical_gap = abs(shoulder_y_end - hip_y_end) # Vertical gap = average shoulder height in last 10 frames - average hip height in last 10 frames
    rule2 = vertical_gap < BODY_HORIZONTAL_THRESHOLD # If the gap between shoulders and hips is lower than the threshold = assume body is horizontal

    # Rule 3: Rapid 10-frame descent
    rule3 = 0
    for i in range(10, len(hip_y)): # Check every 10-frame period
        if (hip_y[i] - hip_y[i-10]) > RAPID_DESCENT_THRESHOLD: # If hip Y descended (increased) by more than the threshold value, Rule 3 = True
            rule3 = 1
            break
    
    # Voting: classify sequence as fall if 2 or more rules are called (True)
    rules_called = [r for r, called in enumerate([rule1, rule2, rule3], start=1) if called]
    prediction  = 1 if len(rules_called) >= 2 else 0

    return prediction, rules_called

def evaluate(dataloader, split_name):
    all_preds  = []
    all_labels = []

    # Rule calling counts
    rule_counts = {1: 0, 2: 0, 3: 0}

    # Iterate through data batches
    for batch in dataloader:
        keypoints = batch['keypoints'].cpu().numpy()
        labels = batch['label'].cpu().numpy()
        
        # Iterate through each sequence in the batch
        for i in range(len(labels)):
            pred, rules_called = classify_sequence(keypoints[i])
            all_preds.append(pred)
            all_labels.append(labels[i])
            for r in rules_called:
                rule_counts[r] += 1

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    print(f"\n{'=' * 55}")
    print(f"  Rule-Based Baseline — {split_name} Set Results")
    print(f"{'=' * 55}")

    print(classification_report(
        all_labels, all_preds,
        target_names=['No Fall', 'Fall'],
        digits=4
    ))

    cm = confusion_matrix(all_labels, all_preds)
    tn, fp, fn, tp = cm.ravel()
    print(f"Confusion Matrix:")
    print(f"  TN={tn}  FP={fp}")
    print(f"  FN={fn}  TP={tp}")

    print(f"\nRule calling counts (out of {len(all_labels)} sequences):")
    for rule, count in rule_counts.items():
        print(f"  Rule {rule}: called {count} times ({count / len(all_labels) * 100:.1f}%)")

    precision, recall, f1, _ = precision_recall_fscore_support(
    all_labels, all_preds, average='binary', zero_division=0)
    print(f"\nOverall F1 (fall class): {f1:.4f}")

# MAIN
# Load dataloaders 
_, val_loader, _ = create_dataloaders(
    split_file='data/dataset_split.json', # Using same split file and normalisation as final LSTM version training
    batch_size=32,
    num_workers=4,
    normalise=True
)

# Evaluate on validation set to tune threshold during development
evaluate(val_loader, split_name="Validation")


Loaded train set:
  Total clips: 5019
  Fall clips: 2280
  No-fall clips: 2739
  Resolution distribution:
    1920x1080: 2353 clips (46.9%)
    1100x1080: 2193 clips (43.7%)
    1280x720: 150 clips (3.0%)
    3840x2160: 130 clips (2.6%)
    848x480: 53 clips (1.1%)
    ... and 10 more resolutions
  Using per-video normalization

Loaded val set:
  Total clips: 1106
  Fall clips: 474
  No-fall clips: 632
  Resolution distribution:
    1920x1080: 523 clips (47.3%)
    1100x1080: 465 clips (42.0%)
    1280x720: 43 clips (3.9%)
    3840x2160: 19 clips (1.7%)
    2336x1080: 13 clips (1.2%)
    ... and 8 more resolutions
  Using per-video normalization

Loaded test set:
  Total clips: 1073
  Fall clips: 457
  No-fall clips: 616
  Resolution distribution:
    1920x1080: 503 clips (46.9%)
    1100x1080: 472 clips (44.0%)
    1280x720: 39 clips (3.6%)
    3840x2160: 22 clips (2.1%)
    848x480: 12 clips (1.1%)
    ... and 6 more resolutions
  Using per-video normalization

  Rule-Based Baseline